# Datatvätt – Telco Customer Churn

Mål: hämta kunddata från Supabase, kontrollera och åtgärda kvalitetsproblem i datan,
och förbereda den för maskininlärning (train/test-split + preprocessing-pipeline).

## 1. Hämta data

Hämtar all data från Supabase-tabellen `telco_churn`. Supabase begränsar
normalt en hämtning till 1000 rader per anrop, så vi paginerar med `.range()`
för att få med samtliga 7043 rader.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(url, key)

all_rows = []
start = 0
batch_size = 1000

while True:
    response = supabase.table("telco_churn").select("*").range(start, start + batch_size - 1).execute()
    if not response.data:
        break
    all_rows.extend(response.data)
    start += batch_size

df = pd.DataFrame(all_rows)
print(len(df))

7043


## 2. Fixa TotalCharges-typen och spara rådata för EDA

`TotalCharges` var sparad som text istället för numeriskt värde. Orsak: nya kunder
(tenure = 0) hade ett tomt mellanslag " " istället för ett tal, vilket gör att
kolumnen inte tolkas som numerisk av pandas. Konverterar med
`pd.to_numeric(errors="coerce")` på hela `df`, en gång, innan vi delar upp datan.

Sparar sedan datan som en lokal fil. Observera att detta bara är en delvis tvätt
(rätt datatyper, men saknade värden och kategoriska kolumner är kvar orörda) —
bra för EDA där man vill se läsbara kategorier och riktiga NaN. Den fullständiga
tvätten (imputering av saknade värden, one-hot-encoding, skalning) sker i steg 4,
i preprocessing-pipelinen, och gäller bara träningsdatan i minnet — den sparas
inte till fil.

In [ ]:
# df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# os.makedirs("data", exist_ok=True)
# df.to_csv("data/telco_churn_clean.csv", index=False)
# print("Sparad:", len(df), "rader")

Sparad: 7043 rader


## 3. Dela upp i features/target och train/test

- Droppar `customerID` (unikt id, ger ingen information till modellen) och `Churn`
  (målvariabeln) från features.
- Omvandlar `Churn` till 1/0 istället för "Yes"/"No" så den kan användas i modellen.
- Delar datan 80/20 i tränings- och testset, stratifierat på `Churn` så andelen
  som churnat är lika stor i båda delarna.

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 4. Bygg preprocessing-pipeline

`TotalCharges` är redan numerisk (fixat i steg 2), så vi bygger direkt
pipelinen som ska förbereda datan för modellträning:
- Numeriska kolumner (`tenure`, `MonthlyCharges`, `TotalCharges`): fyller saknade
  värden med medianen och skalar dem, så alla numeriska variabler blir jämförbara.
- Kategoriska kolumner: fyller saknade värden med vanligaste värdet och
  one-hot-encodar dem, så modellen kan hantera textkategorier.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X_train.columns if c not in num_cols]

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
])

X_train_prepared = preprocessing.fit_transform(X_train)

## 5. Verifiera resultatet

Kontrollerar att pipelinen fungerar som tänkt: rätt antal rader (7043 totalt),
att de kända saknade värdena (TotalCharges) faktiskt fångas upp, och att
outputen har rimlig form och innehåll innan vi går vidare till modellträning.

In [5]:
# Hur mycket saknas egentligen?
print("Antal rader totalt:", len(df))
print(df.isna().sum())
print("\nTotalCharges saknas i X_train:", X_train["TotalCharges"].isna().sum(), "av", len(X_train))

# Se vad pipelinen faktiskt producerar
print("\nForm på output:", X_train_prepared.shape)

feature_names = preprocessing.get_feature_names_out()
X_train_prepared_df = pd.DataFrame(
    X_train_prepared, columns=feature_names, index=X_train.index
)
X_train_prepared_df.head()

Antal rader totalt: 7043
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

TotalCharges saknas i X_train: 9 av 5634

Form på output: (5634, 46)


,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__SeniorCitizen_0,cat__SeniorCitizen_1,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,...,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
3739,0.102105,-0.524660,-0.264216,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3151,-0.995406,-0.029732,-0.788595,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4860,1.240264,1.445085,2.052165,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3867,-0.263732,0.282504,-0.174871,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3810,-1.279946,-0.679117,-0.989791,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


## Slutsats

Datan krävde ingen borttagning av dubbletter eller extremvärden, men behövde
typkonvertering av `TotalCharges` och hantering av 9 saknade värden. Den senare
delen (imputering, encoding, skalning) löses av en återanvändbar
preprocessing-pipeline i steg 4. Rådatan med korrekt typ (`data/telco_churn_clean.csv`)
finns sparad lokalt för EDA — den är inte samma sak som pipeline-outputen.